# ExamScheduling — Colab Reproduction Runner

Runs the full benchmark batch for CSC 2400: **13 algorithms × 8 ITC 2007 instances × 3 seeds**. Budget roughly 60–120 minutes on a free-tier Colab CPU runtime. A Pro runtime halves that.

**What this notebook does:**
1. Installs build deps (g++, OR-Tools, pandas, matplotlib, pytest)
2. Clones the repo and builds the headers-only C++ solver
3. Smoke-tests one algorithm on one instance to catch env breakage early
4. Loops every algorithm × every ITC 2007 set × `NUM_SEEDS` seeds, saving per-run JSON into `results/colab_batch/`
5. Aggregates results into a single CSV and regenerates the paper figures
6. Zips `results/` and triggers a browser download

> **If a step fails mid-batch:** the outer loop `continue`s past failures and still writes whatever finished. Check `results/colab_batch/failures.log` afterwards.

## 1. Environment setup

In [ ]:
!apt-get install -y build-essential > /dev/null 2>&1
!pip install -q ortools pulp numpy pandas matplotlib plotly pytest

## 1b. (optional) Mount Google Drive

Run this cell **before** the zip/download cells to auto-backup the result zips to your Drive. Skip if you only want the browser download.

> `files.download(...)` on its own sends the zip to your computer's Downloads folder; it doesn't touch Drive. Mounting here gives you both: the Drive copy survives runtime disconnects, and the browser download still fires.

In [ ]:
import os

try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_BACKUP_DIR = '/content/drive/MyDrive/exam_scheduling_batches'
    os.makedirs(DRIVE_BACKUP_DIR, exist_ok=True)
    os.environ['DRIVE_BACKUP_DIR'] = DRIVE_BACKUP_DIR
    print(f'Drive mounted. Zips will also land in: {DRIVE_BACKUP_DIR}')
except (ImportError, ModuleNotFoundError):
    print('Not running on Colab — skipping Drive mount.')

## 2. Clone the repo

Update `GH_USER` / `REPO` if you forked. The cell is idempotent — re-running after the clone just `cd`s into the checkout.

In [ ]:
import os

GH_USER = 'Dialovos'
REPO = 'exam-scheduling-algos-analyze'

if not os.path.exists(REPO):
    !git clone --depth 1 https://github.com/{GH_USER}/{REPO}.git
%cd {REPO}
!pwd && git log -1 --oneline

## 3. Build the C++ solver + sanity smoke

In [ ]:
!make
print('\n--- smoke: Tabu on set1, seed 42 ---')
!python main.py --dataset instances/exam_comp_set1.exam --algo tabu --seed 42 --no-batch --quiet

## 4. Full batch

Drops results into `results/colab_batch/<algo>_<set>_<seed>/`. Adjust `ALGOS`, `DATASETS`, or `NUM_SEEDS` below to trim scope. The default is the full paper matrix.

CP-SAT (`cpsat`) has a 60 s time limit per run; the heuristics finish in seconds. IP (`ip`) is skipped automatically inside `main.py` when `n > 500`.

In [ ]:
import glob, os, subprocess, time
from pathlib import Path
from concurrent.futures import ProcessPoolExecutor, as_completed

ALGOS = ['greedy', 'tabu', 'kempe', 'sa', 'alns', 'gd', 'abc', 'ga',
         'lahc', 'woa', 'hho', 'cpsat', 'vns']
DATASETS = sorted(glob.glob('instances/exam_comp_set*.exam'))
NUM_SEEDS = 3

# Worker count: every heuristic is single-threaded C++, so pack runs across cores.
# Leave one vCPU free for the Python driver + I/O; CP-SAT inside the cpsat algo
# is internally single-worker here (num_workers=8 is only set by the Python IP path).
MAX_WORKERS = max(1, (os.cpu_count() or 2) - 1)

BATCH_ROOT = Path('results/colab_batch')
BATCH_ROOT.mkdir(parents=True, exist_ok=True)
FAIL_LOG = BATCH_ROOT / 'failures.log'
if FAIL_LOG.exists(): FAIL_LOG.unlink()


def run_one(algo, dataset, seed):
    """Child-process entry: run one (algo, dataset, seed) combination."""
    ds_name = Path(dataset).stem
    run_dir = BATCH_ROOT / f'{algo}_{ds_name}_seed{seed}'
    run_dir.mkdir(exist_ok=True)
    cmd = ['python', 'main.py', '--dataset', dataset, '--algo', algo,
           '--seed', str(seed), '--no-batch', '--quiet',
           '--output', str(run_dir)]
    try:
        r = subprocess.run(cmd, capture_output=True, text=True, timeout=600)
        if r.returncode != 0:
            return (algo, ds_name, seed, 'fail', r.stderr[-500:])
        return (algo, ds_name, seed, 'ok', '')
    except subprocess.TimeoutExpired:
        return (algo, ds_name, seed, 'timeout', '600s')


jobs = [(a, d, s) for a in ALGOS for d in DATASETS
        for s in range(42, 42 + NUM_SEEDS)]
total = len(jobs)
print(f'Launching {total} runs across {MAX_WORKERS} workers '
      f'({len(ALGOS)} algos × {len(DATASETS)} sets × {NUM_SEEDS} seeds)')
print(f'Output: {BATCH_ROOT.resolve()}')

started = time.time()
completed = 0
with ProcessPoolExecutor(max_workers=MAX_WORKERS) as pool:
    futures = {pool.submit(run_one, *j): j for j in jobs}
    for fut in as_completed(futures):
        algo, ds_name, seed, status, msg = fut.result()
        completed += 1
        elapsed = time.time() - started
        eta = elapsed / completed * (total - completed)
        tag = '  ok' if status == 'ok' else f' {status.upper()}'
        print(f'  [{completed:>3}/{total}] {algo:<7} {ds_name:<18} seed={seed}'
              f'  elapsed={elapsed/60:5.1f}m  eta={eta/60:5.1f}m {tag}',
              flush=True)
        if status != 'ok':
            with open(FAIL_LOG, 'a') as f:
                f.write(f'{algo}/{ds_name}/seed{seed}: {status}\n{msg}\n\n')

print(f'\nDone in {(time.time()-started)/60:.1f} min. Failures:',
      FAIL_LOG.stat().st_size if FAIL_LOG.exists() else 0, 'bytes')

## 5. Aggregate + regenerate figures

Walks `results/colab_batch/`, assembles one long-form DataFrame, runs the full matplotlib plot suite into `results/colab_batch/figures/`.

In [ ]:
import json, pandas as pd
from pathlib import Path
from utils.plots import generate_all_plots

BATCH_ROOT = Path('results/colab_batch')
rows = []
for run_dir in sorted(BATCH_ROOT.iterdir()):
    if not run_dir.is_dir(): continue
    sb = run_dir / 'soft_breakdown.json'
    if not sb.exists(): continue
    # Run dirs look like `{algo}_{dataset}_seed{N}`; dataset can contain
    # underscores (e.g. `exam_comp_set1`), so peel the seed suffix first.
    head, _, seed_tag = run_dir.name.rpartition('_seed')
    if not seed_tag:
        continue  # not a standard run dir
    algo, _, ds_name = head.partition('_')
    try:
        seed = int(seed_tag)
    except ValueError:
        continue
    breakdown = json.loads(sb.read_text())
    for algo_label, parts in breakdown.items():
        row = {'algorithm': algo_label, 'dataset': ds_name, 'seed': seed,
               'soft_penalty': sum(parts.values())}
        row.update(parts)
        rows.append(row)

df = pd.DataFrame(rows)
print(f'Aggregated {len(df)} rows')
df.to_csv(BATCH_ROOT / 'aggregated.csv', index=False)

if not df.empty:
    # Minimum columns for plotting — synthesise runtime + feasibility from the CSV
    df['runtime'] = df.get('runtime', 0.0)
    df['feasible'] = df.get('feasible', True)
    generate_all_plots(df, output_dir=str(BATCH_ROOT / 'figures'))

## 6. Zip + download

Colab sandboxes are ephemeral — do this before the runtime disconnects.

In [ ]:
import os, shutil
from google.colab import files

!zip -r batch_colab.zip results/colab_batch/ > /dev/null

# If Drive was mounted earlier, auto-backup the zip there too.
drive_dir = os.environ.get('DRIVE_BACKUP_DIR', '')
if drive_dir and os.path.isdir(drive_dir):
    shutil.copy('batch_colab.zip', drive_dir)
    print(f'Drive copy: {drive_dir}/batch_colab.zip')

files.download('batch_colab.zip')

## 7. Post-export IP — CP-SAT with Tabu warm-start

> **Run this only after cell #6 finishes downloading `batch_colab.zip`.**
> If the runtime dies mid-IP, the main batch is already safe on disk and Drive.

This cell reuses Tabu solutions from `results/colab_batch/` as a CP-SAT warm-start hint (`add_hint()`). On feasible starting points that's typically a 5–20× speedup over cold CP-SAT. Runs **one instance at a time** because CP-SAT parallelises internally across all vCPUs (`--ip-workers 0` = all cores); outer parallelism would thrash.

- `IP_TIME`: wall-clock per instance. 5 minutes is a reasonable default; raise it on A100 if you want tighter bounds.
- `IP_MAX_EXAMS`: instances larger than this are skipped (default 900 covers 6 of 8 ITC sets).
- Results land in `results/colab_batch_ip/ip_<setname>/` and zip separately as `batch_colab_ip.zip`.

In [ ]:
import glob, subprocess, time
from pathlib import Path

IP_TIME = 300          # seconds per instance
IP_MAX_EXAMS = 900     # skip instances larger than this

main_batch = Path('results/colab_batch')
ip_batch = Path('results/colab_batch_ip')
ip_batch.mkdir(parents=True, exist_ok=True)
ip_fail = ip_batch / 'failures.log'
if ip_fail.exists(): ip_fail.unlink()

datasets = sorted(glob.glob('instances/exam_comp_set*.exam'))
t0 = time.time()

for ds in datasets:
    ds_name = Path(ds).stem
    # Pick the first available Tabu solution as warm-start (any seed works)
    warm_candidates = sorted(main_batch.glob(
        f'tabu_{ds_name}_seed*/solutions/solution_tabu_search_*.sln'))
    warm = str(warm_candidates[0]) if warm_candidates else None

    run_dir = ip_batch / f'ip_{ds_name}'
    run_dir.mkdir(exist_ok=True)
    cmd = ['python', 'main.py', '--dataset', ds, '--algo', 'ip',
           '--ip-time', str(IP_TIME),
           '--ip-max-exams', str(IP_MAX_EXAMS),
           '--ip-workers', '0',                 # 0 = all cores
           '--seed', '42', '--no-batch', '--quiet',
           '--output', str(run_dir)]
    if warm:
        cmd += ['--ip-warmstart', warm]

    print(f'\n=== IP on {ds_name} '
          f'(warm-start: {"yes" if warm else "no"}) ===', flush=True)
    try:
        subprocess.run(cmd, timeout=IP_TIME + 180, check=False)
    except subprocess.TimeoutExpired:
        with open(ip_fail, 'a') as f:
            f.write(f'{ds_name}: TIMEOUT {IP_TIME + 180}s\n\n')
        print(f'  TIMEOUT', flush=True)
    print(f'  total elapsed: {(time.time()-t0)/60:.1f}m', flush=True)

print(f'\nIP batch complete in {(time.time()-t0)/60:.1f}m.')

## 8. Zip + download IP results

Separate zip so you can choose to download just this piece. If Drive is mounted, it also lands there automatically.

In [ ]:
import os, shutil
from google.colab import files

!zip -r batch_colab_ip.zip results/colab_batch_ip/ > /dev/null

drive_dir = os.environ.get('DRIVE_BACKUP_DIR', '')
if drive_dir and os.path.isdir(drive_dir):
    shutil.copy('batch_colab_ip.zip', drive_dir)
    print(f'Drive copy: {drive_dir}/batch_colab_ip.zip')

files.download('batch_colab_ip.zip')

## 9. Scaling batch — dense synthetic ladder (50 → 1000 exams, step 50)

Runs every algorithm on a 20-step synthetic ladder so the runtime-vs-size curve is smooth across the full range. Fixed seed 42, `competition` preset. Files land in `instances/synthetic_scaling_{50..1000}.exam` (regenerated in place if missing, so this cell is idempotent).

- **Cost**: 13 algos × 20 sizes × `SCALING_SEEDS` seeds (default 3) = 780 runs. About 25–45 min on A100 (11 workers), 60–100 min on CPU High-RAM (7 workers).
- **Set `SCALING_STEP = 25`** for a 40-size ladder (1560 runs) if you want a tighter curve and don't mind doubling the wall-clock.
- **Output**: `results/colab_batch_scaling/` with per-run dirs plus an aggregated CSV keyed by `num_exams`, ready for `plot_scaling`.

In [ ]:
import os, json, time, subprocess
from pathlib import Path
from concurrent.futures import ProcessPoolExecutor, as_completed
from core.generator import generate_synthetic, write_itc2007_format

# Ladder config — step 50 gives 20 sizes for a smooth log-scale scaling curve.
# Drop to step=25 for an even denser curve (doubles run count).
SCALING_STEP = 50
SCALING_SIZES = list(range(SCALING_STEP, 1001, SCALING_STEP))
SCALING_ALGOS = ['greedy', 'tabu', 'kempe', 'sa', 'alns', 'gd', 'abc', 'ga',
                 'lahc', 'woa', 'hho', 'cpsat', 'vns']
SCALING_SEEDS = 3

# Runs are single-threaded C++ (low RAM each). Oversubscribe slightly if
# you've got ~50 GB RAM headroom — CPU is the bottleneck, not memory.
# On 8 vCPUs → 7 workers; A100 → 11. Set explicitly to override.
MAX_WORKERS = max(1, (os.cpu_count() or 2) - 1)

# Generate ladder instances if missing (idempotent — fixed seed).
for n in SCALING_SIZES:
    path = Path(f'instances/synthetic_scaling_{n}.exam')
    if not path.exists():
        p = generate_synthetic(num_exams=n, preset='competition', seed=42)
        write_itc2007_format(p, str(path))
        print(f'  generated {path}')

SCALING_ROOT = Path('results/colab_batch_scaling')
SCALING_ROOT.mkdir(parents=True, exist_ok=True)
SCALE_FAIL = SCALING_ROOT / 'failures.log'
if SCALE_FAIL.exists(): SCALE_FAIL.unlink()


def run_scaling(algo, size, seed):
    ds = f'instances/synthetic_scaling_{size}.exam'
    run_dir = SCALING_ROOT / f'{algo}_n{size}_seed{seed}'
    run_dir.mkdir(exist_ok=True)
    cmd = ['python', 'main.py', '--dataset', ds, '--algo', algo,
           '--seed', str(seed), '--no-batch', '--quiet',
           '--output', str(run_dir)]
    try:
        r = subprocess.run(cmd, capture_output=True, text=True, timeout=900)
        return (algo, size, seed,
                'ok' if r.returncode == 0 else 'fail',
                r.stderr[-400:] if r.returncode else '')
    except subprocess.TimeoutExpired:
        return (algo, size, seed, 'timeout', '900s')


jobs = [(a, n, s) for a in SCALING_ALGOS for n in SCALING_SIZES
        for s in range(42, 42 + SCALING_SEEDS)]
print(f'Scaling: {len(SCALING_ALGOS)} algos × {len(SCALING_SIZES)} sizes × '
      f'{SCALING_SEEDS} seeds → {len(jobs)} runs across {MAX_WORKERS} workers')

started = time.time(); done = 0
with ProcessPoolExecutor(max_workers=MAX_WORKERS) as pool:
    futs = {pool.submit(run_scaling, *j): j for j in jobs}
    for fut in as_completed(futs):
        algo, size, seed, status, msg = fut.result()
        done += 1
        elapsed = time.time() - started
        eta = elapsed / done * (len(jobs) - done)
        tag = '  ok' if status == 'ok' else f' {status.upper()}'
        print(f'  [{done:>4}/{len(jobs)}] {algo:<6} n={size:<5} seed={seed}'
              f'  elapsed={elapsed/60:5.1f}m eta={eta/60:5.1f}m {tag}', flush=True)
        if status != 'ok':
            with open(SCALE_FAIL, 'a') as f:
                f.write(f'{algo}/n={size}/seed{seed}: {status}\n{msg}\n\n')

# Aggregate into a single CSV keyed by num_exams.
import pandas as pd
rows = []
for run_dir in sorted(SCALING_ROOT.iterdir()):
    if not run_dir.is_dir(): continue
    summ = run_dir / 'summary.json'
    if not summ.exists(): continue
    head, _, seed_tag = run_dir.name.rpartition('_seed')
    algo, _, n_tag = head.partition('_n')
    n_exams = int(n_tag)
    for algo_label, rec in json.loads(summ.read_text()).items():
        rows.append({
            'algorithm': algo_label, 'num_exams': n_exams,
            'seed': int(seed_tag),
            'soft_penalty': rec['soft'], 'runtime': rec['runtime'],
            'feasible': rec['feasible'], 'hard': rec['hard'],
        })
df = pd.DataFrame(rows)
df.to_csv(SCALING_ROOT / 'aggregated.csv', index=False)
print(f'\nScaling batch complete: {len(df)} rows → {SCALING_ROOT/"aggregated.csv"}')

## 10. Parameter sweep — 1-D sensitivity on 1000-exam instance

For every algo with a declared search space in `tooling/tuner/search_spaces.py`, pin every knob at its tuned default and vary one knob at a time across `SWEEP_VALUES` points (log-spaced for ranges, uniform int for counts). Runs on the largest synthetic instance so soft-penalty variance doesn't drown in noise.

- **Skipped**: `greedy` (no knobs), `cpsat` and `ip` (only a time-budget knob, which is a different axis), `hho` (not in SEARCH_SPACES).
- **Cost** (10 algos, ~18 knob-axes × 5 values × 3 seeds): about 270 runs. Roughly 15–25 min on A100, 30–50 min on CPU High-RAM.
- **Output**: `results/colab_batch_sweep/sensitivity.csv`, consumed by `utils.plots.tuning.plot_parameter_sensitivity` for per-algo sensitivity curves with mean ± std error bars.

In [ ]:
# Tune these to taste — cost scales linearly with both.
# 5 values × 3 seeds per knob gives a real curve shape with error bars.
SWEEP_DATASET = 'instances/synthetic_scaling_1000.exam'
SWEEP_SEEDS = 3
SWEEP_VALUES = 5

!python -m tooling.param_sweep \
    --dataset {SWEEP_DATASET} \
    --seeds {SWEEP_SEEDS} \
    --values {SWEEP_VALUES} \
    --out results/colab_batch_sweep/sensitivity.csv \
    --runs-dir results/colab_batch_sweep/runs

## 11. Zip + download scaling + sweep

Third (and last) zip — bundles both the scaling ladder results and the sensitivity CSV. As with the other zips, if Drive is mounted it's copied there too.

In [ ]:
import os, shutil
from google.colab import files

!zip -r batch_colab_scaling_sweep.zip results/colab_batch_scaling/ results/colab_batch_sweep/ > /dev/null

drive_dir = os.environ.get('DRIVE_BACKUP_DIR', '')
if drive_dir and os.path.isdir(drive_dir):
    shutil.copy('batch_colab_scaling_sweep.zip', drive_dir)
    print(f'Drive copy: {drive_dir}/batch_colab_scaling_sweep.zip')

files.download('batch_colab_scaling_sweep.zip')

## 12. Chain discovery — extensive tune

Runs the auto-tuner's full pipeline (screen, chain discovery, deep param tuning, chain rescore) across every ITC set and persists the best chain to `tuned_params.json`. CPU-calibrated: about 2.5–3 h on CPU High-RAM (8 vCPU, Pro), 5–6 h on the free CPU tier. Auto-resumes if the runtime disconnects; re-run the cell and it picks up where it left off.

**Chain-finder v2 defaults (2026-04):** max chain length 10, adaptive successive-halving eta, 1-point crossover between survivors (25 %), partial-credit scoring for chains whose last step times out (2.5 % penalty), step-level early-stop on hopeless partials (>2.5× dataset baseline), prefix `.sln` cache to warm-start shared prefixes, no adjacent-duplicate algos. All on by default; roll back individually via the `--no-chain-*` flags below.

Trim scope by lowering `MAX_TIME`, `CHAIN_POP`, or `CHAIN_ROUNDS`, or drop `EVAL_DATASETS` to 2 for a quicker smoke.

In [ ]:
import os, subprocess, time

CHAIN_OUT = 'results/colab_batch_chain'
CHAIN_WORKERS = max(1, (os.cpu_count() or 2) - 1)

# Budget knobs — defaults aim at ~2.5–3 h on CPU High-RAM.
# Chain-finder v2: larger pop works well with adaptive eta (16 → 5 → 2 at
# eta_schedule=[3,2,2]) and the prefix cache absorbs the extra cost by
# amortizing shared prefixes across candidates.
CHAIN_POP     = 16    # chain-tournament population
CHAIN_ROUNDS  = 6     # tournament rounds (successive halving inside each)
CHAIN_MAX_LEN = 10    # v2 default — up from 5
PARAM_TRIALS  = 5     # deep param trials per core algo (Phase 3)
EVAL_DATASETS = 3     # ITC sets used for tuner evaluation (out of 8)
MAX_TIME      = 9600  # total wall-clock budget, seconds (160 min)

cmd = ['python', '-u', 'main.py', '--mode', 'tune',
       '--chain-pop', str(CHAIN_POP), '--chain-rounds', str(CHAIN_ROUNDS),
       '--chain-max-len', str(CHAIN_MAX_LEN),
       '--param-trials', str(PARAM_TRIALS), '--eval-datasets', str(EVAL_DATASETS),
       '--tune-workers', str(CHAIN_WORKERS), '--max-time', str(MAX_TIME),
       '--seed', '42', '--output', CHAIN_OUT, '--no-batch']

# v2 rollback flags — uncomment to disable specific features:
# cmd += ['--no-chain-prefix-cache']     # disable warm-start .sln cache
# cmd += ['--no-chain-partial-credit']   # drop truncated-chain partial scoring
# cmd += ['--no-chain-early-stop']       # keep running hopeless partials to the end
# cmd += ['--chain-allow-duplicates']    # permit adjacent-duplicate algos

# Resume from a prior session's checkpoint if one exists.
ckpt = os.path.join(CHAIN_OUT, 'tuning', 'checkpoint.json')
if os.path.isfile(ckpt):
    cmd.append('--resume')
    print(f'Resuming from {ckpt}')

print('Command:', ' '.join(cmd), flush=True)
print('(output is unbuffered via python -u — you should see each algo/dataset result land live)', flush=True)
started = time.time()
p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                     text=True, bufsize=1)
for line in p.stdout:
    print(line, end='', flush=True)
rc = p.wait()
print(f'\n[chain-tune] rc={rc}  elapsed={(time.time()-started)/60:.1f} min')

## 13. Top-5 chains — dedupe by algo sequence

Reads the tuner's `chain_history` (every chain scored this session) and prints the top-5 by score, deduped so only distinct algorithm pipelines surface. Writes `top5_chains.{json,txt}` into the chain batch for post-session inspection.

In [ ]:
import json
from collections import defaultdict
from pathlib import Path

chain_dir = Path('results/colab_batch_chain')
ckpt_path = chain_dir / 'tuning' / 'checkpoint.json'
if not ckpt_path.is_file():
    raise SystemExit(f'No tuner checkpoint at {ckpt_path} — did section 12 run?')

ckpt = json.loads(ckpt_path.read_text())
history = ckpt.get('chain_history', [])
if not history:
    raise SystemExit('chain_history is empty — chain discovery phase never ran. '
                     'Check the tune log for the reason (likely time budget too short).')

# Dedupe by algo signature, keep best score + eval count.
best_by_sig, counts = {}, defaultdict(int)
for entry in history:
    chain, score = entry[0], entry[1]
    sig = tuple(step[0] for step in chain)
    counts[sig] += 1
    if sig not in best_by_sig or score < best_by_sig[sig][1]:
        best_by_sig[sig] = (chain, score)

feasible = [(c, s) for c, s in best_by_sig.values() if s < 1e6]
ranked = sorted(feasible, key=lambda x: x[1])[:5]

top5 = [{'rank': i+1, 'chain': chain, 'score': float(score),
         'n_evaluations': counts[tuple(s[0] for s in chain)]}
        for i, (chain, score) in enumerate(ranked)]

out = {
    'total_chain_trials': len(history),
    'unique_structures':  len(best_by_sig),
    'feasible_structures': len(feasible),
    'best_chain':        ckpt.get('best_chain'),
    'best_chain_score':  ckpt.get('best_chain_score'),
    'top5': top5,
}
(chain_dir / 'top5_chains.json').write_text(json.dumps(out, indent=2))

lines = [f"Top 5 chains  ({len(history)} evaluations, {len(best_by_sig)} unique structures, "
         f"{len(feasible)} feasible)", '='*78]
for item in top5:
    chain = item['chain']
    lines.append(f"\n#{item['rank']}  score={item['score']:.4f}  "
                 f"evaluated {item['n_evaluations']}x")
    lines.append(f"    {' -> '.join(s[0] for s in chain)}")
    for j, step in enumerate(chain):
        algo, params = step[0], step[1]
        ps = '  '.join(f'{k}={v}' for k, v in sorted(params.items())) if params else '(no params)'
        lines.append(f"      step {j+1}: {algo:<8} {ps}")
bc = ckpt.get('best_chain')
if bc:
    bcs = ckpt.get('best_chain_score')
    bcs_fmt = f'{bcs:.4f}' if isinstance(bcs, (int, float)) else str(bcs)
    lines.append('\n' + '='*78)
    lines.append(f'saved best_chain (score={bcs_fmt}):  '
                 + ' -> '.join(s[0] for s in bc))
text = '\n'.join(lines)
(chain_dir / 'top5_chains.txt').write_text(text)
print(text)


## 14. Zip + download chain batch

Fourth zip (`batch_colab_chain.zip`) — tuner checkpoint + best_chain + top-5 log + `tuned_params.json`. Drive-backed if the mount cell ran.

In [ ]:
import os, shutil
from google.colab import files

extras = 'tuned_params.json' if os.path.isfile('tuned_params.json') else ''
!zip -r batch_colab_chain.zip results/colab_batch_chain/ {extras} > /dev/null

drive_dir = os.environ.get('DRIVE_BACKUP_DIR', '')
if drive_dir and os.path.isdir(drive_dir):
    shutil.copy('batch_colab_chain.zip', drive_dir)
    print(f'Drive copy: {drive_dir}/batch_colab_chain.zip')

files.download('batch_colab_chain.zip')
